In [12]:
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

In [9]:
from dotenv import load_dotenv
load_dotenv(override=1)

True

In [10]:
index_name = "langchain-test"

In [33]:
pc.delete_index('langchain-test')

In [43]:
index_name+ f' index {212}'

'langchain-test index 212'

In [48]:
pc.list_indexes().names()

['langchain-test', 'lfk']

In [52]:
# Initialize Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
for i in range(10):
    # Create index if it does not exist
    name  = index_name+ f'-index-{i}'
    if name not in pc.list_indexes().names():
        pc.create_index(
            name=name,
            dimension=3072,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
    print('done')
    
    # index = pc.Index(index_name)

done
done
done


ForbiddenException: (403)
Reason: Forbidden
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '1b32559ec690295754af397a15081daf', 'date': 'Sat, 14 Mar 2026 11:17:29 GMT', 'server': 'Google Frontend', 'Content-Length': '257', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"FORBIDDEN","message":"Request failed. You've reached the max serverless indexes allowed in project Default (5). Use namespaces to partition your data into logical groups, or upgrade your plan to add more serverless indexes."},"status":403}


In [35]:
# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

In [36]:
test = embeddings.embed_query('test')

In [37]:
len(test)

3072

In [38]:
# Connect LangChain to Pinecone
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [39]:
# Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for AI."),
    Document(page_content="Vector databases store embeddings for similarity search.")
]

In [40]:
# Add documents to Pinecone
vectorstore.add_documents(docs)

['0bc6b064-cadd-4a7b-8921-981be112cbfc',
 '68930e1a-49ce-4a8e-90e6-6a487caeeeeb',
 '1faaae18-ef0d-4535-a43c-56c34aa52b26']

In [41]:
# Query test
query = "What is Pinecone?"

results = vectorstore.similarity_search(query, k=2)

for r in results:
    print(r.page_content)

Pinecone is a vector database for AI.
Vector databases store embeddings for similarity search.
